# Fine-tune ConvNeXt note classifier

In [13]:
# Colab setup. Run after uploading/cloning the full project folder.
# !pip install -r requirements.txt

from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "nsynth_pipeline").is_dir():
            return path
    raise FileNotFoundError("Could not find nsynth_pipeline. Upload/clone the full project, then run from inside it.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [14]:
import sys
print(sys.executable)

/usr/bin/python3


In [15]:
import sys, torch

print("Python:", sys.executable)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Python: /usr/bin/python3
CUDA: True
GPU: Tesla T4


In [17]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

from nsynth_pipeline.convexnet import freeze_module, load_convnext_backbone, unfreeze_last_blocks
from nsynth_pipeline.data import MelConfig, NOTE_NAMES, make_nsynth_loaders
from nsynth_pipeline.models import ConvNeXtNoteClassifier
from nsynth_pipeline.config import PITCH_LOSS_WEIGHT
from nsynth_pipeline.train_utils import move_batch, save_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

ModuleNotFoundError: No module named 'nsynth_pipeline'

In [ ]:
DATASET_NAME = "jg583/NSynth"
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS_FROZEN = 5
EPOCHS_UNFROZEN = 3
UNDERPERFORMING_ACCURACY = 0.80

# Set these to small integers for smoke tests; keep None for full train/test splits.
MAX_TRAIN_ITEMS = None
MAX_TEST_ITEMS = None

CONVNEXT_VARIANT = "tiny"
# Use None for no download, or "DEFAULT" for torchvision pretrained weights.
CONVNEXT_WEIGHTS = None

CHECKPOINT_PATH = "checkpoints/convnext_note_classifier.pt"
mel_config = MelConfig()

In [ ]:
# In Colab, this stores the Hugging Face dataset cache on Google Drive so it persists across runtimes.
HF_CACHE_DIR = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    from nsynth_pipeline.config import HF_CACHE_DIR as DRIVE_HF_CACHE_DIR
    HF_CACHE_DIR = DRIVE_HF_CACHE_DIR
except ModuleNotFoundError:
    print("Not running in Colab; using the local Hugging Face cache.")

print("HF cache dir:", HF_CACHE_DIR)

In [ ]:
loaders = make_nsynth_loaders(
    dataset_name=DATASET_NAME,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    max_train_items=MAX_TRAIN_ITEMS,
    max_test_items=MAX_TEST_ITEMS,
    mel_config=mel_config,
    cache_dir=HF_CACHE_DIR,
)
batch = next(iter(loaders["train"]))
batch["spectrogram"].shape, batch["note_class"][:8], [NOTE_NAMES[i] for i in batch["note_class"][:8].tolist()]

In [ ]:
backbone = load_convnext_backbone(variant=CONVNEXT_VARIANT, weights=CONVNEXT_WEIGHTS)
freeze_module(backbone)

model = ConvNeXtNoteClassifier(backbone).to(device)
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)

## Training code

In [ ]:
def train_classifier_one_epoch(model, loader, optimizer, device):
    model.train()
    if not any(parameter.requires_grad for parameter in model.backbone.parameters()):
        model.backbone.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="classifier train"):
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)

        logits = model(batch)
        loss = F.cross_entropy(logits, batch["note_class"])
        loss.backward()
        optimizer.step()

        batch_size = batch["note_class"].shape[0]
        running_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=-1) == batch["note_class"]).sum().item()
        total += batch_size

    return {"loss": running_loss / total, "accuracy": correct / total}


@torch.no_grad()
def evaluate_classifier(model, loader, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="classifier eval"):
        batch = move_batch(batch, device)
        logits = model(batch)
        loss = F.cross_entropy(logits, batch["note_class"])

        batch_size = batch["note_class"].shape[0]
        running_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=-1) == batch["note_class"]).sum().item()
        total += batch_size

    return {"loss": running_loss / total, "accuracy": correct / total}

In [ ]:
best_accuracy = 0.0
for epoch in range(1, EPOCHS_FROZEN + 1):
    train_metrics = train_classifier_one_epoch(model, loaders["train"], optimizer, device)
    test_metrics = evaluate_classifier(model, loaders["test"], device)
    best_accuracy = max(best_accuracy, test_metrics["accuracy"])
    print({"epoch": epoch, "train": train_metrics, "test": test_metrics})
    save_checkpoint(CHECKPOINT_PATH, model_state_dict=model.state_dict(), mel_config=mel_config, note_names=NOTE_NAMES)

best_accuracy

In [ ]:
if best_accuracy < UNDERPERFORMING_ACCURACY:
    print(f"Accuracy {best_accuracy:.3f} is below {UNDERPERFORMING_ACCURACY:.3f}; unfreezing the last ConvNeXt block.")
    unfreeze_last_blocks(model.backbone, blocks=1)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
    for epoch in range(1, EPOCHS_UNFROZEN + 1):
        train_metrics = train_classifier_one_epoch(model, loaders["train"], optimizer, device)
        test_metrics = evaluate_classifier(model, loaders["test"], device)
        best_accuracy = max(best_accuracy, test_metrics["accuracy"])
        print({"unfrozen_epoch": epoch, "train": train_metrics, "test": test_metrics})
        save_checkpoint(CHECKPOINT_PATH, model_state_dict=model.state_dict(), mel_config=mel_config, note_names=NOTE_NAMES)
else:
    print("Backbone stayed frozen.")